[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai-forever/FRIDA-Decisions/blob/main/examples/support_triage/support_triage.ipynb)

# Route support tickets by confidence

A ticket has to reach a team, and someone has to decide when the model should hand it to a person. An LLM would write a label you then parse. FRIDA-Decisions scores the team (`choice`), the urgency (`score`) and two yes/no checks in one encoder pass, and returns probabilities. The threshold that sends a ticket to an operator lives in code: changing it does not rewrite the questions.

Option ids are short and neutral (`auth`, `billing`). The model sees `"<id>: <description>"`, so the meaning sits in the description.

Patterns: [Intent routing](https://docs.typesafe.ai/patterns/intent-routing) and [Confidence-gated routing](https://docs.typesafe.ai/patterns/confidence-routing). `confidence` is reported for `choice` and `score`, not for `noul`.


In [1]:
%pip install -q "frida-decisions[torch] @ git+https://github.com/ai-forever/FRIDA-Decisions@v0.4.0"


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import torch
import frida_decisions
from frida_decisions import Judge

MODEL = os.environ.get("FD_MODEL", "ai-forever/FRIDA-Decisions")   # repo id or a local folder
device = "cuda" if torch.cuda.is_available() else "cpu"
judge = Judge.from_pretrained(MODEL, device=device)
gpu = f" ({torch.cuda.get_device_name(0)})" if device == "cuda" else ""
print(f"frida-decisions {frida_decisions.__version__}, loaded on {device}{gpu}, {judge.dtype}")


frida-decisions 0.4.0, loaded on cpu, torch.float32


The questions and the routing rule. Urgency and the two yes/no checks set the priority only. They do not pick the team. `0.7` is a starting confidence; the sweep below is the measurement.


In [3]:
DATA = [
    {"team": "auth", "text": "Не могу войти в личный кабинет: пишет, что пароль неверный, а письмо для сброса не приходит."},
    {"team": "billing", "text": "Вчера оплатил заказ картой, деньги списались дважды. Верните, пожалуйста, лишнее списание."},
    {"team": "shipping", "text": "Курьер должен был привезти заказ вчера, трек-номер не обновляется второй день."},
    {"team": "product", "text": "Чайник пришёл с трещиной на колбе, кипятить его нельзя. Хочу обмен на такой же."},
    {"team": "billing", "text": "Подписка продлилась сама, я её не продлял. Снимите списание за этот месяц."},
    {"team": "auth", "text": "Поменял номер телефона и теперь не могу подтвердить вход: код приходит на старый номер."},
    {"team": "product", "text": "В коробке нет зарядки, которая указана в комплекте. Сама доставка пришла вовремя, не хватает вещи."},
    {"team": "product", "text": "Наушники хрипят на левом канале с первого дня. В описании этого не было."},
    {"team": "billing", "text": "Чек на сайте показывает одну сумму, а в банке списалось больше. Пришлите правильный чек."},
    {"team": "shipping", "text": "Нужно изменить адрес доставки: курьер ещё не выехал, заказ только собирают на складе."},
    {"team": "auth", "text": "После смены почты не приходит ни код входа, ни уведомления. Доступа к старому ящику больше нет."},
    {"team": "shipping", "text": "Курьер опоздал на три часа и даже не позвонил. Больше заказывать у вас не буду."},
]

TEAMS = {
    "auth": "доступ к аккаунту, вход, пароль, личные данные",
    "billing": "оплата, списания, возвраты денег, подписки",
    "shipping": "доставка, курьер, сроки и адрес получения заказа",
    "product": "качество, комплектность и характеристики товара",
}
URGENCY = [
    "Очень низкая: работа не страдает, ответить можно когда угодно.",
    "Низкая: неудобство небольшое, ответить можно на этой неделе.",
    "Средняя: работа затруднена, но обходной путь есть.",
    "Высокая: работа заблокирована, обходного пути нет.",
    "Критическая: потери растут с каждым часом.",
]
DEFAULT_CONFIDENCE = 0.7
HIGH_URGENCY = 3.0
UPSET = 0.6


def request_for(text):
    return {
        "state": text,
        "questions": {
            "team": {"type": "choice", "instructions": "В какую команду направить обращение?",
                     "criteria": TEAMS},
            "urgency": {"type": "score", "instructions": "Оцени срочность обращения по шкале ниже.",
                        "criteria": URGENCY},
            "angry": {"type": "noul", "instructions": "Автор раздражён?"},
            "churn": {"type": "noul", "instructions": "Автор грозит уйти или больше не покупать?"},
        },
    }


def route(answers, min_confidence):
    team = answers["team"]
    urgent = answers["urgency"]["score"] >= HIGH_URGENCY
    upset = answers["angry"]["noul"] >= UPSET or answers["churn"]["noul"] >= UPSET
    auto = team["confidence"] >= min_confidence
    decision = {"auto": auto, "confidence": team["confidence"]}
    if auto:
        decision["queue"] = team["choice"]
    else:
        decision["queue"] = "operator"
    if urgent or upset:
        decision["priority"] = "high"
    else:
        decision["priority"] = "normal"
    return decision


def coverage(scored, min_confidence):
    """Accuracy is only over tickets the model routes itself."""
    auto = 0
    correct = 0
    for row in scored:
        decision = route(row["answers"], min_confidence)
        if not decision["auto"]:
            continue
        auto += 1
        if decision["queue"] == row["team"]:
            correct += 1
    return auto, correct


Every ticket is scored once. Coverage is the share sent to a team without a person. A ticket handed to an operator is not counted as a miss.


In [4]:
def run(judge):
    responses = judge.judge_batch([request_for(row["text"]) for row in DATA])
    scored = []
    print(f"{'result':8} {'queue':10} {'pri':6} conf  urgency angry churn  gold")
    for row, response in zip(DATA, responses):
        answers = response["answers"]
        decision = route(answers, DEFAULT_CONFIDENCE)
        scored.append({"team": row["team"], "answers": answers})
        if not decision["auto"]:
            mark = "hold"
        elif decision["queue"] == row["team"]:
            mark = "ok"
        else:
            mark = "miss"
        print(f"{mark:8} {decision['queue']:10} {decision['priority']:6} "
              f"{decision['confidence']:.2f}  {answers['urgency']['score']:.2f}    "
              f"{answers['angry']['noul']:.2f}  {answers['churn']['noul']:.2f}   {row['team']}")

    print(f"\nthreshold {DEFAULT_CONFIDENCE}:")
    auto, correct = coverage(scored, DEFAULT_CONFIDENCE)
    if auto:
        accuracy = f"{correct}/{auto}"
    else:
        accuracy = "n/a"
    print(f"  auto {auto}/{len(scored)} ({auto / len(scored):.0%}), accuracy on auto {accuracy}")
    print("\nthreshold sweep:")
    for threshold in (0.0, 0.5, 0.7, 0.8, 0.9, 0.95):
        auto, correct = coverage(scored, threshold)
        if auto:
            accuracy = f"{correct / auto:.0%}"
        else:
            accuracy = "n/a"
        print(f"  {threshold:.2f}  coverage {auto / len(scored):.0%}  accuracy {accuracy}")


In [5]:
run(judge)


result   queue      pri    conf  urgency angry churn  gold
ok       auth       normal 1.00  2.59    0.25  0.09   auth
ok       billing    normal 0.98  1.28    0.30  0.17   billing
ok       shipping   normal 0.95  2.32    0.47  0.15   shipping
ok       product    normal 0.99  1.04    0.26  0.21   product
ok       billing    normal 1.00  1.07    0.08  0.07   billing
ok       auth       normal 0.95  2.19    0.25  0.15   auth
ok       product    high   0.91  0.80    0.67  0.43   product
ok       product    normal 0.99  1.21    0.55  0.41   product
ok       billing    normal 0.79  1.04    0.30  0.29   billing
ok       shipping   normal 1.00  1.46    0.05  0.11   shipping
ok       auth       normal 1.00  2.92    0.16  0.13   auth
ok       shipping   high   0.78  0.96    0.67  0.52   shipping

threshold 0.7:
  auto 12/12 (100%), accuracy on auto 12/12

threshold sweep:
  0.00  coverage 100%  accuracy 100%
  0.50  coverage 100%  accuracy 100%
  0.70  coverage 100%  accuracy 100%
  0.80  covera